In [ ]:
"""
This script generates the source data for Supplementary Figure 7, which will show the topographies of Alpha and Beta Low/High sub-bands power.
"""

import pandas as pd
import numpy as np
import mne
import os
import sys

datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Data directory: {datadir}')

maindir = '' # this is the directory with the BIDS project folder

pipver = ''
task = 'rest'
phases = ['p2', 'p5']

lfreq = 0.1 #Hz
hfreq = 145.0 #Hz
fsample = 300.0 #Hz
frange = f"{round(lfreq, 1)}-{int(hfreq)}Hz"

trans = True # Whether to use head transformation or not
zmm = 44 # destination z coordinate head position in mm

fitting_param = 'finley'
megtype = 'grad'
icselection = 'ecg04eog08' # 'allbutecg04' #'eog08' #
proc = 'filt' + icselection #'sss' #'clean'

# Prepare df for barplots
# Directories and file names
deriv_folder = f'aperiodic_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'
taskref = 'rest'
phaseref = 'p5'
armref = 1
bids_project_folder = f'BIDS_long_{phaseref}_{taskref}_arm{armref}'
deriv_root = os.path.join(maindir, bids_project_folder,
                          'derivatives', deriv_folder)
statsdir = os.path.join(deriv_root, 'stats')


# ---- FUNCTIONS ----
# --- Function to get mean values for each variable and channel, averaged across phases for each subject ---
def get_varmean(datafile, megtype):
    df = pd.read_csv(datafile, sep='\t').set_index(['row'])
    df = df.drop(columns=['task', 'Age0', 'deltaAge'])
    df2 = df[df.phase == 'p2']
    df2 = df2.set_index('subject')
    df2.drop(columns=['phase'], inplace=True)
    df5 = df[df.phase == 'p5']
    df5 = df5.set_index('subject')
    df5.drop(columns=['phase'], inplace=True)
    df = pd.concat([df2, df5]).groupby('subject').mean()
    if megtype == 'grad':
        channels = [c[:-1] + '1' for c in df.columns if c.startswith('MEG') and (c.endswith('2'))]
        tmpdf = pd.DataFrame(columns=channels)
        for c in channels:
            tmpdf[c] = df[[f'{c[:-1]}2', f'{c[:-1]}3']].mean(axis=1)
        df = pd.concat([df, tmpdf], axis=1)
        df = df.drop(columns=[c for c in df.columns if c.endswith('2') or c.endswith('3')])
    else:
        channels = [c for c in df.columns if c.startswith('MEG') and (c.endswith('1'))]

    count = df.count()
    mean = df.mean()
    return mean, count, channels

# --- Function to get channel positions from layout file ---
def get_channels_positions(channels):
    path_to_fieldtrip = '/path/to/fieldtrip'  # Update this path to your FieldTrip installation
    # Load standard positions for the channels
    layoutdir = os.path.join(path_to_fieldtrip, 'template', 'layout')
    layout = mne.channels.read_layout(os.path.join(layoutdir, 'neuromag306mag.lay'))
    
    # --- Preparation for topographic plots --- 
    # get the positions of the channels
    pos = []                        
    for ch in channels:
        if ch in layout.names:
            pos.append(layout.pos[layout.names.index(ch),0:2]/5)
        else:
            raise ValueError(f'Channel {ch} not found in layout')

    pos = np.array(pos) 
    return pos


# --- End of functions ----
# ------------------------------------------------------------------------------

####################################################################################################

parameters = ['low_alpha_band_power', 'high_alpha_band_power', 'low_beta_band_power', 'high_beta_band_power']

# Loop over parameters and create bar plots and topographies
for i, parameter in enumerate(parameters):
    col_y = f'{parameter}_{megtype}'

    # Topography for each parameter
    # Datafile for each megtype and variable of interest,
    # containing all the subjects, phases, age, and channels
    datafile = os.path.join(
        statsdir, f'aperiodic_stier_{proc}_{fitting_param}_{megtype}{parameter}_2betas.tsv'
    )     
    varmean, count, channels = get_varmean(datafile, megtype)

    pos = get_channels_positions(channels)
    pos -= 0.1
    pos =  (pos*1.2)
    pos[:,1] = pos[:,1] + 0.014 
    pos[:,0] = pos[:,0] + 0.007

    topovals = varmean[channels].to_numpy(float)

    df_topo_save = pd.DataFrame({
        'Channel': channels,
        'Mean_Value': topovals,
        'Positions': list(pos),
    })

    topo_outfile = os.path.join(datadir, f'supp_figure07_{parameter}_{megtype}_topo.tsv')
    df_topo_save.to_csv(topo_outfile, sep='\t', index=False)
    print(f"Saved topography mean data to: {topo_outfile}")

Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure07_low_alpha_band_power_grad_topo.tsv
Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure07_high_alpha_band_power_grad_topo.tsv
Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure07_low_beta_band_power_grad_topo.tsv
Saved topography mean data to: /imaging/camcan/sandbox/mc06/data/supp_figure07_high_beta_band_power_grad_topo.tsv
